# EuroSAT input-normalization ablation

This notebook measures how input-band normalization affects frozen EuroSAT representations. It keeps the checkpoint and embedding definition fixed at `pre_norm + mean_fine`, then compares statistics from MMEarth pretraining, the EuroSAT training split, and GEO-Bench's packaged statistics.

Normalization is selected using **training and validation data only**. The official test split is constructed only after the strategy has been selected.

## Context & Methods

The model was pretrained with per-band MMEarth z-scores. The current GEO-Bench adapter instead uses the dataset-level `band_stats.json`. These transformations need not be equivalent because sensor processing level, geography, season, and dataset construction change the per-band distributions.

| Strategy | Mean and standard deviation | Role |
|---|---|---|
| `mmearth_pretrain` | Exact MMEarth `sentinel2_l1c` statistics | Deployment-compatible primary candidate |
| `eurosat_train` | Computed from official EuroSAT training images only | Dataset-adapted candidate without validation/test leakage |
| `geobench_official` | GEO-Bench packaged `band_stats.json` | Reproduces the repository's current benchmark preprocessing |
| `no_zscore` | Raw values, mean 0 and standard deviation 1 | Sanity control; excluded from selection |

### Key assumptions

- EuroSAT's raw Sentinel-2 values and MMEarth's L1C values use compatible units. The per-band diagnostic tables test this assumption.
- Invalid GEO-Bench pixels are non-finite values, matching the current adapter.
- The encoder stays frozen and routing is deterministic.
- Every strategy uses the same samples, embedding, probe hyperparameters, and seeds.
- A 0.5 percentage-point validation interval defines a practical tie. Ties favor MMEarth statistics because they require no downstream-dataset adaptation.

## 1. Parameters

Use complete splits for the final ablation. For a smoke test, set the sample limits and `stats_sample_limit` to small values.

In [ ]:
from pathlib import Path
import os

repo_root = Path.cwd().resolve()
if repo_root.name == 'notebooks':
    repo_root = repo_root.parent

default_run_dir = repo_root
run_dir = Path(os.environ.get('MEOX_RUN_DIR', default_run_dir))
config_path = Path(os.environ.get(
    'MEOX_CONFIG', repo_root / 'configs/pretrain_mmearth_moe_mae_full.yaml'
))
checkpoint_path = Path(os.environ.get(
    'MEOX_CHECKPOINT', repo_root / 'weights/pretrained/meox_s_mmearth64_best.pth'
))
geobench_root = Path(os.environ['GEO_BENCH_DIR'])
cache_dir = run_dir / 'analysis/geobench_eurosat_input_normalization'

dataset_name = 'm-eurosat'
partition_name = 'default'
mmearth_s2_stats_key = 'sentinel2_l1c'
token_source = 'pre_norm'
pooling = 'mean_fine'

train_limit = None
valid_limit = None
test_limit = None
stats_sample_limit = None
quantile_values_per_image = 128
batch_size = 128
num_workers = 4
sample_seed = 42
device_override = None
reuse_cache = True

probe_epochs = 50
probe_batch_size = 1024
probe_learning_rate = 5e-2
probe_seeds = (0, 1, 2, 3, 4)
tie_tolerance = 0.005
winner_override = None  # Example: 'mmearth_pretrain'
evaluate_selected_on_test = True

## 2. Load the checkpoint and raw selection splits

In [ ]:
import json
import random
import sys
from copy import deepcopy

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset, TensorDataset
from tqdm.auto import tqdm

sys.path.insert(0, str(repo_root))

from datasets.geobench import GeoBenchClassificationDataset
from datasets.mmearth import MMEarthDataset
from utils.extract_embeddings import (
    build_model_from_config, load_config, make_inference_dataloader,
    maybe_limit_dataset, model_input_schema, move_to_device, resolve_device,
)
from utils.linear_probe import classification_metrics

assert config_path.exists(), f'Missing config: {config_path}'
assert checkpoint_path.exists(), f'Missing checkpoint: {checkpoint_path}'
assert (geobench_root / 'classification_v1.0' / dataset_name).exists(), (
    f'Missing GEO-Bench EuroSAT data under {geobench_root}'
)

config = load_config(str(config_path))
device = resolve_device(device_override)
_, model_band_names, _ = model_input_schema(config)

def build_raw_split(split_name):
    return GeoBenchClassificationDataset(
        root_dir=str(geobench_root), dataset_name=dataset_name, split=split_name,
        model_band_names=model_band_names, partition_name=partition_name,
    )

train_base = build_raw_split('train')
valid_base = build_raw_split('valid')
model = build_model_from_config(config, str(checkpoint_path), device)

assert list(train_base.band_mapping) == ['sentinel2'], train_base.band_mapping
source_names = [source for source, _ in train_base.band_mapping['sentinel2']]
target_names = [target for _, target in train_base.band_mapping['sentinel2']]
assert target_names == model_band_names['sentinel2']

print(f'device={device} train={len(train_base)} valid={len(valid_base)}')
print('source bands:', source_names)
print('model bands:', target_names)
print('source mapping:', train_base.band_mapping)
print('preprocessing:', train_base.preprocessing_signature)
print('checkpoint:', checkpoint_path)

## 3. Compute raw EuroSAT statistics

Means and standard deviations are computed with streaming sums over valid pixels. Quantiles use a bounded deterministic pixel sample. Training statistics define the `eurosat_train` normalization; validation statistics are diagnostic only.

In [ ]:
def pack_raw_sample(sample):
    array, _ = sample.pack_to_3d(
        band_names=source_names, resample=True, resample_order=0
    )
    array = array.astype(np.float32, copy=False)
    return array, np.isfinite(array)

def raw_array(dataset_info, index):
    return pack_raw_sample(dataset_info.dataset[index])

def selected_stat_indices(length, limit, seed):
    if limit is None or limit >= length:
        return np.arange(length)
    if limit <= 0:
        raise ValueError('stats_sample_limit must be positive')
    return np.random.default_rng(seed).choice(length, size=limit, replace=False)

def compute_raw_statistics(dataset_info, limit, seed):
    indices = selected_stat_indices(len(dataset_info), limit, seed)
    channels = len(source_names)
    counts = np.zeros(channels, dtype=np.int64)
    sums = np.zeros(channels, dtype=np.float64)
    squared_sums = np.zeros(channels, dtype=np.float64)
    sampled_values = [[] for _ in range(channels)]
    rng = np.random.default_rng(seed)

    for index in tqdm(indices, desc=f'Raw statistics ({dataset_info.dataset.split})'):
        array, validity = raw_array(dataset_info, int(index))
        for channel in range(channels):
            values = array[..., channel][validity[..., channel]].astype(np.float64)
            counts[channel] += values.size
            sums[channel] += values.sum()
            squared_sums[channel] += np.square(values).sum()
            sample_size = min(quantile_values_per_image, values.size)
            if sample_size:
                selected = rng.choice(values.size, size=sample_size, replace=False)
                sampled_values[channel].append(values[selected])

    means = sums / counts
    variances = np.maximum(squared_sums / counts - np.square(means), 0.0)
    samples = [np.concatenate(values) for values in sampled_values]
    return {
        'count': counts, 'mean': means, 'std': np.sqrt(variances),
        'p01': np.asarray([np.percentile(values, 1) for values in samples]),
        'median': np.asarray([np.median(values) for values in samples]),
        'p99': np.asarray([np.percentile(values, 99) for values in samples]),
    }

def load_or_compute_raw_statistics(split_name, dataset_info, limit):
    cache_dir.mkdir(parents=True, exist_ok=True)
    limit_tag = 'full' if limit is None else str(limit)
    cache_path = cache_dir / (
        f'{dataset_name}_{split_name}_{dataset_info.preprocessing_signature}_'
        f'raw_stats_{limit_tag}.npz'
    )
    if reuse_cache and cache_path.exists():
        with np.load(cache_path, allow_pickle=False) as archive:
            print('loaded:', cache_path)
            return {name: archive[name] for name in archive.files}
    stats = compute_raw_statistics(dataset_info, limit, sample_seed)
    np.savez_compressed(cache_path, **stats)
    print('saved:', cache_path)
    return stats

In [ ]:
train_raw_stats = load_or_compute_raw_statistics(
    'train', train_base, stats_sample_limit
)
valid_raw_stats = load_or_compute_raw_statistics(
    'valid', valid_base, stats_sample_limit
)
raw_stats_table = pd.DataFrame({
    'band': target_names,
    'train_mean': train_raw_stats['mean'],
    'train_std': train_raw_stats['std'],
    'train_p01': train_raw_stats['p01'],
    'train_median': train_raw_stats['median'],
    'train_p99': train_raw_stats['p99'],
    'valid_mean': valid_raw_stats['mean'],
    'valid_std': valid_raw_stats['std'],
})
display(raw_stats_table.set_index('band').round(3))

## 4. Resolve the four normalization strategies

The MMEarth statistics file is resolved from the same pretraining configuration that builds the checkpoint. The comparison table should be inspected before running the probe: very large normalized means or standard deviations indicate incompatible raw units or the wrong MMEarth processing level.

In [ ]:
def resolve_mmearth_stats_path(active_config):
    training = active_config['training']
    subset = training.get('dataset_subset', training.get('subset', 'MMEarth64'))
    dataset_name_for_subset = MMEarthDataset.subsets[subset]
    root = Path(training['dataset_path'])
    direct = root / f'{dataset_name_for_subset}_band_stats.json'
    nested = root / dataset_name_for_subset / f'{dataset_name_for_subset}_band_stats.json'
    if direct.exists():
        return direct
    if nested.exists():
        return nested
    raise FileNotFoundError(f'Cannot resolve MMEarth band statistics from {root}')

mmearth_stats_path = resolve_mmearth_stats_path(config)
with open(mmearth_stats_path, 'r', encoding='utf-8') as handle:
    mmearth_all_stats = json.load(handle)
if mmearth_s2_stats_key not in mmearth_all_stats:
    raise KeyError(
        f'{mmearth_s2_stats_key} is absent from {mmearth_stats_path}; '
        f'available keys: {sorted(mmearth_all_stats)}'
    )
mmearth_s2_stats = mmearth_all_stats[mmearth_s2_stats_key]
mmearth_indices = [
    MMEarthDataset.all_modality_bands['sentinel2'].index(name) for name in target_names
]

geobench_means = np.asarray(
    [train_base.dataset.band_stats[name].mean for name in source_names], dtype=np.float64
)
geobench_stds = np.asarray(
    [train_base.dataset.band_stats[name].std for name in source_names], dtype=np.float64
)
normalization_stats = {
    'mmearth_pretrain': {
        'mean': np.asarray(mmearth_s2_stats['mean'], dtype=np.float64)[mmearth_indices],
        'std': np.asarray(mmearth_s2_stats['std'], dtype=np.float64)[mmearth_indices],
    },
    'eurosat_train': {
        'mean': train_raw_stats['mean'].astype(np.float64),
        'std': train_raw_stats['std'].astype(np.float64),
    },
    'geobench_official': {'mean': geobench_means, 'std': geobench_stds},
    'no_zscore': {
        'mean': np.zeros(len(target_names), dtype=np.float64),
        'std': np.ones(len(target_names), dtype=np.float64),
    },
}
for strategy, stats in normalization_stats.items():
    if np.any(~np.isfinite(stats['mean'])) or np.any(stats['std'] <= 0):
        raise ValueError(f'Invalid statistics for {strategy}')

strategy_order = [
    'mmearth_pretrain', 'eurosat_train', 'geobench_official', 'no_zscore'
]
selection_strategies = strategy_order[:-1]
selection_priority = [
    'mmearth_pretrain', 'eurosat_train', 'geobench_official'
]
print('MMEarth statistics:', mmearth_stats_path)
print('MMEarth S2 processing level:', mmearth_s2_stats_key)

In [ ]:
comparison_rows = []
for band_index, band in enumerate(target_names):
    for strategy in strategy_order:
        mean = normalization_stats[strategy]['mean'][band_index]
        std = normalization_stats[strategy]['std'][band_index]
        comparison_rows.append({
            'band': band, 'strategy': strategy, 'normalizer_mean': mean,
            'normalizer_std': std,
            'normalized_train_mean': (train_raw_stats['mean'][band_index] - mean) / std,
            'normalized_train_std': train_raw_stats['std'][band_index] / std,
            'normalized_valid_mean': (valid_raw_stats['mean'][band_index] - mean) / std,
            'normalized_valid_std': valid_raw_stats['std'][band_index] / std,
        })
normalization_table = pd.DataFrame(comparison_rows)
display(normalization_table.round(3))

fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)
x = np.arange(len(target_names))
for strategy in strategy_order:
    rows = normalization_table[normalization_table['strategy'] == strategy]
    axes[0].plot(x, rows['normalized_valid_mean'], marker='o', label=strategy)
    axes[1].plot(x, rows['normalized_valid_std'], marker='o', label=strategy)
axes[0].axhline(0.0, color='black', linewidth=1, alpha=0.5)
axes[1].axhline(1.0, color='black', linewidth=1, alpha=0.5)
axes[0].set_ylabel('Validation mean after normalization')
axes[1].set_ylabel('Validation std after normalization')
axes[1].set_xticks(x, target_names)
axes[0].legend(ncol=2)
axes[0].grid(alpha=0.2)
axes[1].grid(alpha=0.2)
fig.suptitle('Input distribution under each normalization strategy')
plt.tight_layout()
plt.show()

## 5. Extract fixed embeddings

Only the input statistics change. The data order, model weights, deterministic routing, token source, and pooling remain fixed. The no-z-score control runs in float32 to avoid half-precision overflow from raw digital values.

In [ ]:
class NormalizedEuroSatDataset(Dataset):
    def __init__(self, dataset_info, means, stds):
        self.dataset_info = dataset_info
        self.means = np.asarray(means, dtype=np.float32).reshape(1, 1, -1)
        self.stds = np.asarray(stds, dtype=np.float32).reshape(1, 1, -1)
        self.raster_band_names = dataset_info.raster_band_names

    def __len__(self):
        return len(self.dataset_info)

    def __getitem__(self, index):
        sample = self.dataset_info.dataset[index]
        array, validity = pack_raw_sample(sample)
        normalized = (array - self.means) / self.stds
        normalized = np.where(validity, normalized, 0.0).astype(np.float32)
        return {
            'raster_dict': {
                'sentinel2': torch.from_numpy(np.moveaxis(normalized, -1, 0).copy())
            },
            'raster_valid_masks': {
                'sentinel2': torch.from_numpy(np.moveaxis(validity, -1, 0).copy())
            },
            'label': torch.as_tensor(sample.label, dtype=torch.long),
            'sample_id': sample.sample_name,
        }

def build_normalized_dataset(dataset_info, strategy, limit):
    stats = normalization_stats[strategy]
    dataset = NormalizedEuroSatDataset(dataset_info, stats['mean'], stats['std'])
    return maybe_limit_dataset(dataset, limit, sample_seed)

In [ ]:
@torch.inference_mode()
def extract_strategy_embeddings(dataset, strategy):
    dataloader = make_inference_dataloader(dataset, batch_size, num_workers)
    embeddings = []
    labels = []
    sample_ids = []
    model.eval()
    use_amp = device.type == 'cuda' and strategy != 'no_zscore'

    for batch in tqdm(dataloader, desc=f'Embeddings ({strategy})'):
        rasters = move_to_device(batch['raster_dict'], device)
        validity = move_to_device(batch['raster_valid_masks'], device)
        with torch.amp.autocast(device_type=device.type, enabled=use_amp):
            features = model.forward_features(
                raster_dict=rasters, raster_valid_masks=validity,
                raster_band_names=train_base.raster_band_names,
                return_routing=False, stochastic_routing=False,
            )
            values = model.encoder._pool_feature_tokens(
                features, token_source=token_source, pooling=pooling
            )
        embeddings.append(values.float().cpu())
        labels.append(batch['label'].cpu())
        sample_ids.extend(str(value) for value in batch['sample_id'])

    return {
        'embeddings': torch.cat(embeddings).numpy(),
        'labels': torch.cat(labels).numpy(),
        'sample_ids': np.asarray(sample_ids, dtype=str),
    }

def load_or_extract_embeddings(split_name, dataset_info, strategy, limit):
    cache_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_stat = checkpoint_path.stat()
    checkpoint_tag = (
        f'{checkpoint_path.stem}_{checkpoint_stat.st_size}_{checkpoint_stat.st_mtime_ns}'
    )
    limit_tag = 'full' if limit is None else str(limit)
    cache_path = cache_dir / (
        f'{dataset_name}_{split_name}_{dataset_info.preprocessing_signature}_'
        f'{checkpoint_tag}_{strategy}_{limit_tag}.npz'
    )
    if reuse_cache and cache_path.exists():
        with np.load(cache_path, allow_pickle=False) as archive:
            print('loaded:', cache_path)
            return {name: archive[name] for name in archive.files}
    dataset = build_normalized_dataset(dataset_info, strategy, limit)
    outputs = extract_strategy_embeddings(dataset, strategy)
    np.savez_compressed(cache_path, **outputs)
    print('saved:', cache_path)
    return outputs

In [ ]:
train_outputs = {}
valid_outputs = {}
for strategy in strategy_order:
    train_outputs[strategy] = load_or_extract_embeddings(
        'train', train_base, strategy, train_limit
    )
    valid_outputs[strategy] = load_or_extract_embeddings(
        'valid', valid_base, strategy, valid_limit
    )
    assert np.isfinite(train_outputs[strategy]['embeddings']).all()
    assert np.isfinite(valid_outputs[strategy]['embeddings']).all()
    print(
        strategy, train_outputs[strategy]['embeddings'].shape,
        valid_outputs[strategy]['embeddings'].shape,
    )

## 6. Embedding geometry diagnostic

These metrics describe distribution changes but do not select the normalization. Validation linear-probe accuracy remains the selection criterion.

In [ ]:
def mean_pairwise_cosine(values):
    normalized = values / np.clip(np.linalg.norm(values, axis=1, keepdims=True), 1e-12, None)
    count = len(normalized)
    return float((np.square(normalized.sum(axis=0)).sum() - count) / (count * (count - 1)))

def effective_rank(values):
    centered = values - values.mean(axis=0, keepdims=True)
    singular_values = np.linalg.svd(centered, compute_uv=False)
    eigenvalues = np.square(singular_values)
    probabilities = eigenvalues / np.clip(eigenvalues.sum(), 1e-12, None)
    probabilities = probabilities[probabilities > 0]
    return float(np.exp(-(probabilities * np.log(probabilities)).sum()))

geometry_rows = []
for strategy in strategy_order:
    values = valid_outputs[strategy]['embeddings'].astype(np.float64)
    norms = np.linalg.norm(values, axis=1)
    geometry_rows.append({
        'strategy': strategy, 'mean_embedding_norm': norms.mean(),
        'std_embedding_norm': norms.std(),
        'raw_pairwise_cosine': mean_pairwise_cosine(values),
        'centered_effective_rank': effective_rank(values),
    })
geometry_table = pd.DataFrame(geometry_rows).set_index('strategy')
display(geometry_table.round(4))

## 7. Validation-only linear-probe ablation

A fresh linear layer is trained for every strategy and seed. The best epoch is selected by validation average accuracy. The no-z-score control is displayed but cannot be selected automatically.

In [ ]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

@torch.inference_mode()
def predict_logits(head, features):
    batches = DataLoader(
        torch.from_numpy(features).float(), batch_size=probe_batch_size
    )
    return torch.cat([head(batch.to(device)).cpu() for batch in batches]).numpy()

def train_strategy_probe(strategy, train_data, valid_data, test_data=None):
    train_features = train_data['embeddings']
    valid_features = valid_data['embeddings']
    train_labels = train_data['labels'].astype(np.int64).reshape(-1)
    valid_labels = valid_data['labels'].astype(np.int64).reshape(-1)
    num_classes = int(max(train_labels.max(), valid_labels.max()) + 1)
    probe_dataset = TensorDataset(
        torch.from_numpy(train_features).float(), torch.from_numpy(train_labels).long()
    )
    rows = []

    for seed in probe_seeds:
        seed_everything(int(seed))
        head = nn.Linear(train_features.shape[1], num_classes).to(device)
        optimizer = torch.optim.AdamW(
            head.parameters(), lr=probe_learning_rate, weight_decay=0.0
        )
        generator = torch.Generator().manual_seed(int(seed))
        train_loader = DataLoader(
            probe_dataset, batch_size=probe_batch_size, shuffle=True, generator=generator
        )
        best_score = float('-inf')
        best_epoch = 0
        best_metrics = None
        best_state = None

        for epoch in range(1, probe_epochs + 1):
            head.train()
            for features, labels in train_loader:
                logits = head(features.to(device))
                loss = nn.functional.cross_entropy(logits, labels.to(device))
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()
            head.eval()
            valid_logits = predict_logits(head, valid_features)
            metrics = classification_metrics(valid_logits, valid_labels, multilabel=False)
            if metrics['average_accuracy'] > best_score:
                best_score = metrics['average_accuracy']
                best_epoch = epoch
                best_metrics = metrics
                best_state = deepcopy(head.state_dict())

        row = {
            'strategy': strategy, 'seed': int(seed), 'best_epoch': int(best_epoch),
            **{f'valid_{name}': value for name, value in best_metrics.items()},
        }
        if test_data is not None:
            head.load_state_dict(best_state)
            head.eval()
            test_labels = test_data['labels'].astype(np.int64).reshape(-1)
            test_metrics = classification_metrics(
                predict_logits(head, test_data['embeddings']), test_labels, multilabel=False
            )
            row.update({f'test_{name}': value for name, value in test_metrics.items()})
        rows.append(row)
    return rows

In [ ]:
validation_rows = []
for strategy in tqdm(strategy_order, desc='Normalization probes'):
    validation_rows.extend(
        train_strategy_probe(strategy, train_outputs[strategy], valid_outputs[strategy])
    )
validation_runs = pd.DataFrame(validation_rows)
validation_summary = (
    validation_runs.groupby('strategy', sort=False)
    .agg(
        mean_valid_aa=('valid_average_accuracy', 'mean'),
        std_valid_aa=('valid_average_accuracy', 'std'),
        mean_valid_oa=('valid_overall_accuracy', 'mean'),
        mean_valid_macro_f1=('valid_macro_f1', 'mean'),
        mean_best_epoch=('best_epoch', 'mean'),
    )
    .reset_index()
    .sort_values('mean_valid_aa', ascending=False)
    .reset_index(drop=True)
)
validation_summary.index = validation_summary.index + 1
validation_summary.index.name = 'rank'
display(validation_summary.round(4))

cache_dir.mkdir(parents=True, exist_ok=True)
normalization_table.to_csv(cache_dir / 'input_normalization_diagnostics.csv', index=False)
geometry_table.to_csv(cache_dir / 'embedding_geometry.csv')
validation_runs.to_csv(cache_dir / 'validation_runs.csv', index=False)
validation_summary.to_csv(cache_dir / 'validation_summary.csv', index=True)

In [ ]:
eligible_summary = validation_summary[
    validation_summary['strategy'].isin(selection_strategies)
]
best_mean = eligible_summary['mean_valid_aa'].max()
tied_strategies = [
    strategy for strategy in selection_priority
    if strategy in set(
        eligible_summary.loc[
            eligible_summary['mean_valid_aa'] >= best_mean - tie_tolerance, 'strategy'
        ]
    )
]
if winner_override is not None:
    if winner_override not in selection_strategies:
        raise ValueError(f'Unknown or ineligible winner_override: {winner_override}')
    selected_strategy = winner_override
else:
    selected_strategy = tied_strategies[0]

print(f'Best eligible validation mean AA: {best_mean:.4f}')
print(f'Tied within {tie_tolerance:.3f}: {tied_strategies}')
print('Selected strategy:', selected_strategy)

plot_table = validation_summary.sort_values('mean_valid_aa')
colors = [
    '#C84B31' if strategy == selected_strategy else '#176B87'
    for strategy in plot_table['strategy']
]
fig, axis = plt.subplots(figsize=(9, 5))
axis.barh(
    plot_table['strategy'], plot_table['mean_valid_aa'],
    xerr=plot_table['std_valid_aa'].fillna(0), color=colors, capsize=3,
)
axis.axvline(best_mean - tie_tolerance, color='#555555', linestyle='--', linewidth=1)
axis.set_xlabel('Validation average accuracy')
axis.set_title('EuroSAT input-normalization ablation')
axis.grid(axis='x', alpha=0.2)
plt.tight_layout()
plt.show()

display(validation_runs.pivot(
    index='seed', columns='strategy', values='valid_average_accuracy'
).round(4))

## 8. Final test evaluation

The official test split is loaded only now, after `selected_strategy` has been fixed. Only the selected strategy is extracted and evaluated.

In [ ]:
if evaluate_selected_on_test:
    test_base = build_raw_split('test')
    test_outputs = load_or_extract_embeddings(
        'test', test_base, selected_strategy, test_limit
    )
    final_runs = pd.DataFrame(train_strategy_probe(
        selected_strategy, train_outputs[selected_strategy],
        valid_outputs[selected_strategy], test_outputs,
    ))
    test_columns = [
        'test_average_accuracy', 'test_overall_accuracy', 'test_macro_f1'
    ]
    final_summary = pd.DataFrame({
        'metric': test_columns,
        'mean': [final_runs[column].mean() for column in test_columns],
        'std': [final_runs[column].std(ddof=1) for column in test_columns],
    })
    display(final_runs.round(4))
    display(final_summary.set_index('metric').round(4))
    final_runs.to_csv(cache_dir / 'selected_strategy_test_runs.csv', index=False)
    result_manifest = {
        'checkpoint': str(checkpoint_path.resolve()),
        'preprocessing_signature': train_base.preprocessing_signature,
        'embedding': f'{token_source}__{pooling}',
        'selected_strategy': selected_strategy,
        'mmearth_stats_path': str(mmearth_stats_path.resolve()),
        'mmearth_s2_stats_key': mmearth_s2_stats_key,
        'selection_metric': 'validation_average_accuracy',
        'tie_tolerance': tie_tolerance,
        'probe_epochs': probe_epochs,
        'probe_learning_rate': probe_learning_rate,
        'probe_seeds': list(probe_seeds),
        'split_sizes': {
            'train': len(train_outputs[selected_strategy]['labels']),
            'valid': len(valid_outputs[selected_strategy]['labels']),
            'test': len(test_outputs['labels']),
        },
        'test_aggregate': {
            row['metric']: {'mean': float(row['mean']), 'std': float(row['std'])}
            for _, row in final_summary.iterrows()
        },
    }
    with open(cache_dir / 'selected_strategy_results.json', 'w', encoding='utf-8') as handle:
        json.dump(result_manifest, handle, indent=2)
else:
    print('Test evaluation disabled. Selection is complete:', selected_strategy)

## 9. Interpretation checklist

1. Confirm that MMEarth and EuroSAT raw means and standard deviations are in compatible numerical units.
2. Check whether MMEarth normalization leaves extreme per-band means or scales on validation data. If so, verify the L1C/L2A key before interpreting probe performance.
3. Use validation average accuracy, not embedding geometry or the test result, to rank normalization strategies.
4. Treat differences below 0.5 percentage points as practically tied unless additional seeds establish a stable separation.
5. Report `mmearth_pretrain` and `eurosat_train` as controlled normalization ablations, not as alternative primary protocols.
6. `eurosat_train` is dataset-adapted preprocessing and cannot be used as the general foundation-model inference path.
7. `geobench_official` reproduces the primary downstream adapter, but its packaged statistics are split-independent and should not be described as train-only statistics.
8. The downstream protocol is now frozen to `geobench_official` input normalization and `pre_norm` + `mean_fine` embeddings. This notebook does not reopen normalization, token-source, or pooling selection.